# 19 — NeMo Guardrails with the same Ollama model
Input and output self-check rails are advisory safety screens. Pydantic,
citation checks, and server-side approval remain the hard boundaries.



**Advanced follow-up to lesson 9 (control).**


## Problem

A model can receive hostile instructions or return an unsafe draft.


## Concept

NeMo self-check rails use the configured Ollama model as an opt-in screen.

NeMo checks input/output without replacing CrewAI generation. Its Ollama engine uses the configured local model.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call   ◀ YOU ARE HERE
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email   ◀ YOU ARE HERE

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
input → NeMo/Ollama input rail → CrewAI → NeMo output rail → Python approval gate
```

question → NeMo input check → CrewAI → NeMo output check → policy gate. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Inspect the shared Ollama configuration

Inspect the shared Ollama configuration.


In [ ]:
import os
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.common import model_name
from workshop.nemo_guardrails import check_input, check_output, enabled

print("NeMo enabled:", enabled())
print("Configured model:", model_name())



## Run a local-model check when NeMo is enabled
This is opt-in because the self-check invokes the local model twice.



## Step 2 — Run input and output checks only when the local model is ready

Run input and output checks only when the local model is ready.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    if not enabled():
        raise RuntimeError("Set WORKSHOP_NEMO_ENABLED=true for the live guardrail lab")
    question = "Should Northstar AI pilot Meridian AI?"
    check_input(question)
    check_output(question, '{"recommendation":"pilot","rationale":"verify security claims"}')
    print("Input and output rails passed")
else:
    print("Enable NeMo and WORKSHOP_RUN_LIVE_DEMOS=true for model-backed checks.")


## Failure mode to know about

A self-check can miss an attack or block a benign question. The model never authorizes a side effect.


## Production implication

Pin and evaluate NeMo on the target local model; define timeouts and retain hard policy checks.


## Your turn

Try a prompt-injection fixture and inspect the check result.

**Checkpoint:** Model-backed rails are visible and never replace deterministic authorization.
